In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

from vivarium import Artifact, InteractiveContext
import pandas as pd, numpy as np, os

In [ ]:
!whoami
!date

# V&V neonatal mortality in an interactive simulation

General approach:
* Check quantities subject to stochastic uncertainty both visually and with a statistical test
* Single draw, location, and scenario (baseline)
* Only test relative to artifact, because GBD shared functions can't be directly used in the sim environment -- **this notebook will not find issues in artifact creation**

In [ ]:
%pip list | grep vivarium

In [ ]:
%pip freeze | grep vivarium

In [ ]:
! cat /mnt/share/homes/zmbc/src/vivarium_gates_mncnh/.git/HEAD

In [ ]:
draw_num = 60

In [ ]:
from pathlib import Path

In [ ]:
import vivarium_gates_mncnh
from vivarium.framework.configuration import build_model_specification

main_sim_model_specification = build_model_specification(
    Path(vivarium_gates_mncnh.__file__).parent / 'model_specifications/model_spec.yaml'
)
del main_sim_model_specification.configuration.observers
main_sim_model_specification.configuration.input_data.input_draw_number = draw_num
main_sim_model_specification.configuration.population.population_size = 10_000_000

In [ ]:
paf_sim_model_specification = build_model_specification(
    Path(vivarium_gates_mncnh.__file__).parent / 'data/lbwsg_paf/code/lbwsg_paf.yaml'
)
# Don't delete observers since we use these to get the PAFs
paf_sim_model_specification.configuration.input_data.input_draw_number = draw_num
paf_sim_model_specification.configuration.population.population_size = (
    400**2 # 400x400 grid on gestational age and birth weight...
    * 58 # ... in each LBWSG category
    * 2 # ... for each sex
)

In [ ]:
location = "Ethiopia"

In [ ]:
orig_location = Path(main_sim_model_specification.configuration.input_data.artifact_path).stem
assert orig_location == Path(paf_sim_model_specification.configuration.input_data.artifact_path).stem
orig_location

In [ ]:
main_sim_model_specification.configuration.input_data.artifact_path = main_sim_model_specification.configuration.input_data.artifact_path.replace(orig_location, location.lower())
assert Path(main_sim_model_specification.configuration.input_data.artifact_path).stem == location.lower()
paf_sim_model_specification.configuration.input_data.artifact_path = paf_sim_model_specification.configuration.input_data.artifact_path.replace(orig_location, location.lower())
assert Path(paf_sim_model_specification.configuration.input_data.artifact_path).stem == location.lower()

In [ ]:
art = Artifact(main_sim_model_specification.configuration.input_data.artifact_path)

## Create simulations

In [ ]:
%%time

main_sim = InteractiveContext(main_sim_model_specification)

In [ ]:
main_sim_components = main_sim.list_components()
list(main_sim_components.keys())

In [ ]:
%%time

paf_sim = InteractiveContext(paf_sim_model_specification)

In [ ]:
paf_sim_components = paf_sim.list_components()
list(paf_sim_components.keys())

## ENN mortality

### Step main sim to ENN mortality

In [ ]:
get_event_name = main_sim._builder.time.simulation_event_name()
get_event_name()

In [ ]:
%%time

while get_event_name() != 'early_neonatal_mortality':
    main_sim.step()
    print(get_event_name())

In [ ]:
assert get_event_name() == 'early_neonatal_mortality'

### Check that PAF sim is in ENN

In [ ]:
from vivarium_gates_mncnh.constants.data_values import LATE_NEONATAL_AGE_START, LATE_NEONATAL_AGE_END

assert (paf_sim.get_population(["child_age"]).child_age < LATE_NEONATAL_AGE_START).all()

### Transfer PAFs and preterm prevalence to main sim

In [ ]:
from vivarium.framework.event import Event

In [ ]:
paf_sim._results.gather_results(Event(
    "time_step",
    index=paf_sim.get_population_index(),
    user_data={},
    time=paf_sim._clock.time,
    step_size=paf_sim._clock.step_size,
))

In [ ]:
def process_pafs(pafs):
    pafs = pafs.rename(columns={"child_sex": "sex_of_child"})
    pafs["child_age_start"] = pafs["child_age_group"].map({
        "early_neonatal": 0,
        "late_neonatal": LATE_NEONATAL_AGE_START,
    })
    pafs["child_age_end"] = pafs["child_age_group"].map({
        "early_neonatal": LATE_NEONATAL_AGE_START,
        "late_neonatal": LATE_NEONATAL_AGE_END,
    })
    pafs["year_start"] = 2021
    pafs["year_end"] = 2022

    return pafs.drop(columns=["child_age_group"])

In [ ]:
acmrisk_pafs = process_pafs(
    paf_sim.get_results()['calculated_lbwsg_paf_on_cause.all_causes.all_cause_mortality_risk']
)
acmrisk_pafs

In [ ]:
preterm_csmr_pafs = process_pafs(
    paf_sim.get_results()['calculated_lbwsg_paf_on_cause.all_causes.all_cause_mortality_risk_preterm']
)
preterm_csmr_pafs

In [ ]:
import vivarium

def update_lookup_table(lookup_table, new_data):
    # NOTE: When we get to Vivarium 4.1 we can just call lookup_table.set_data(new_data)
    lookup_table.data = new_data
    # https://github.com/ihmeuw/vivarium/blob/c4d88f7c76df9650f426c43fd44131d5bb272709/src/vivarium/framework/lookup/table.py#L93-L115
    parameter_columns_with_edges: list[tuple[str, str, str]] = [
        (p, f"{p}_start", f"{p}_end") for p in lookup_table.parameter_columns
    ]

    lookup_table.interpolation = vivarium.framework.lookup.table.Interpolation(
        lookup_table.data,
        lookup_table.key_columns,
        parameter_columns_with_edges,
        lookup_table.value_columns,
        order=lookup_table._manager.interpolation_order,
        extrapolate=lookup_table._manager.extrapolate,
        validate=lookup_table._manager.validate_interpolation,
    )

In [ ]:
acmr_paf_components = [k for k in main_sim_components if 'risk_effect.low_birth_weight_' in k and 'preterm' not in k]
acmr_paf_components

In [ ]:
for component in acmr_paf_components:
    update_lookup_table(main_sim_components[component].paf_table, acmrisk_pafs)

In [ ]:
preterm_paf_components = [k for k in main_sim_components if ('risk_effect.low_birth_weight_' in k and 'preterm' in k) or ('preterm_birth.' in k)]
preterm_paf_components

In [ ]:
for component in preterm_paf_components:
    update_lookup_table(main_sim_components[component].paf_table, preterm_csmr_pafs)

### Check all-cause mortality risk

In [ ]:
# Check that our transfer above actually worked
assert (
    set(main_sim.get_population("all_causes.all_cause_mortality_risk.paf").unique())
    <=
    set(acmrisk_pafs['value'].unique())
)

In [ ]:
# https://github.com/ihmeuw/vivarium_gates_mncnh/blob/9502fba51790f522523b8cd2925e539a692babf6/src/vivarium_gates_mncnh/components/mortality.py#L281-L283
pop = main_sim.get_population([
    "child_alive",
    "sex_of_child",
    "pregnancy_outcome",
    "effect_of_low_birth_weight_and_short_gestation_on_early_neonatal_all_causes_relative_risk",
    "effect_of_low_birth_weight_and_short_gestation_on_late_neonatal_all_causes_relative_risk"
])
alive_idx = pop.index[pop['child_alive'] == True]
mortality_risk = main_sim.get_population("death_in_age_group_probability").loc[alive_idx]

In [ ]:
def get_acmrisk_targets(age_group_start, age_group_end):
    return (
        art.load('cause.all_causes.all_cause_mortality_risk')[f'draw_{draw_num}'].reset_index()
        .pipe(lambda df: df[(df.age_start == age_group_start) & (df.age_end == age_group_end)])
        .drop(columns=['age_start', 'age_end', 'year_start', 'year_end'])
        .set_index('sex')
        [f'draw_{draw_num}']
    )

In [ ]:
enn_acmrisk_targets = get_acmrisk_targets(0, LATE_NEONATAL_AGE_START)
enn_acmrisk_targets

In [ ]:
from vivarium_testing_utils.fuzzy_checker import FuzzyChecker
import matplotlib.pyplot as plt

fuzzy_checker = FuzzyChecker()
any_failures = False

def fuzzy_check(observed_values, targets, name, acceptable_deviation=None):
    global any_failures
    print('Targets')
    display(targets)

    observed_value_summaries = observed_values.groupby(pop.sex_of_child).describe()

    print('Summaries of observed values')
    display(observed_value_summaries)

    # if acceptable_deviation is None:
    #     target_lambda = lambda x: x
    # else:
    #     target_lambda = lambda x: (x * (1 - acceptable_deviation), x * (1 + acceptable_deviation))

    # for sex in ['Female', 'Male']:
    #     try:
    #         fuzzy_checker.fuzzy_assert_mean(
    #             observed_values=observed_values[pop.sex_of_child == sex],
    #             target_mean=target_lambda(targets.loc[sex]),
    #             name=f'{name} by sex',
    #         )
    #     except AssertionError as e:
    #         print(e)
    #         any_failures = True

    # try:
    #     overall_target = (pop.loc[alive_idx].groupby('sex_of_child').size() * targets).sum() / len(alive_idx)
    #     fuzzy_checker.fuzzy_assert_mean(
    #         observed_values=observed_values,
    #         target_mean=target_lambda(overall_target),
    #         name=name,
    #     )
    # except AssertionError as e:
    #     print(e)
    #     any_failures = True

    plt.errorbar(
        targets.loc[observed_value_summaries.index],
        observed_value_summaries['mean'],
        # Frequentist SE for plots
        yerr=observed_value_summaries['std'] / np.sqrt(observed_value_summaries['count']),
        fmt="o",
    )
    max_val = max(targets.loc[observed_value_summaries.index].max(), observed_value_summaries['mean'].max())
    min_val = min(targets.loc[observed_value_summaries.index].min(), observed_value_summaries['mean'].min())
    plt.plot([min_val, max_val], [min_val, max_val], 'k--')
    plt.title(f'{name} by sex')
    plt.xlabel('artifact value')
    plt.ylabel('simulation value')
    plt.show()

    plt.errorbar(
        targets.loc[observed_value_summaries.index],
        observed_value_summaries['mean'] / targets.loc[observed_value_summaries.index],
        # Frequentist SE for plots
        yerr=observed_value_summaries['std'] / np.sqrt(observed_value_summaries['count']) / targets.loc[observed_value_summaries.index],
        fmt="o",
    )
    plt.plot([min_val, max_val], [1, 1], 'k--')
    plt.title(f'{name} by sex relative error')
    plt.xlabel('artifact value')
    plt.ylabel('simulation value / artifact value')
    plt.show()

In [ ]:
fuzzy_check(mortality_risk, enn_acmrisk_targets, "early neonatal all-cause mortality risk")

#### Check individual steps in calculation of all-cause mortality risk

In [ ]:
def step_by_step_check_acmrisk(age_group_start, age_group_end):
    acmrisk_targets = get_acmrisk_targets(age_group_start, age_group_end)
    # https://github.com/ihmeuw/vivarium_gates_mncnh/blob/86f8e36e8f781549cd6671cba9ad33912638dc4a/src/vivarium_gates_mncnh/components/mortality.py#L357-L361
    print('Initial mortality risk')
    initial_all_cause_mortality_risk = main_sim_components['neonatal_mortality'].all_cause_mortality_risk(alive_idx)
    # Does not vary except by sex
    assert (initial_all_cause_mortality_risk.groupby(pop.sex_of_child).nunique() == 1).all()
    # Exactly matches artifact values
    assert np.allclose(initial_all_cause_mortality_risk.groupby(pop.sex_of_child).mean(), acmrisk_targets, rtol=0, atol=1e-14)
    display(initial_all_cause_mortality_risk.groupby(pop.sex_of_child).mean())

    # Next (conceptually) is applying LBWSG PAF and RR
    lbwsg_paf = main_sim_components['risk_effect.low_birth_weight_and_short_gestation_on_cause.all_causes.all_cause_mortality_risk'].paf_table(alive_idx)
    if age_group_start == 0 and age_group_end == LATE_NEONATAL_AGE_START:
        age_group_name = 'early_neonatal'
    elif age_group_start == LATE_NEONATAL_AGE_START and age_group_end == LATE_NEONATAL_AGE_END:
        age_group_name = 'late_neonatal'
    else:
        raise ValueError("Unknown age group")

    lbwsg_rr = pop.loc[alive_idx][f'effect_of_low_birth_weight_and_short_gestation_on_{age_group_name}_all_causes_relative_risk']

    acmrisk_after_lbwsg = initial_all_cause_mortality_risk * (1 - lbwsg_paf) * lbwsg_rr
    print('After LBWSG:')
    display(acmrisk_after_lbwsg.groupby(pop.sex_of_child).describe())

    fuzzy_check(acmrisk_after_lbwsg, acmrisk_targets, "ACMRisk after LBWSG")

    # Check that we've exactly replicated everything up to mutators
    pop_mgr = main_sim._builder.population._manager
    pipeline = pop_mgr._get_attribute_pipelines()["death_in_age_group_probability"]
    assert np.allclose(acmrisk_after_lbwsg, pipeline.source(pop_mgr, alive_idx), rtol=0, atol=1e-14)

    # Now we add in CSMRisk-based modifications
    print('Mutators:')
    display([m.name for m in pipeline.mutators])

    working_acmrisk = acmrisk_after_lbwsg

    for mutator in pipeline.mutators:
        print(f'After {mutator.name}')
        working_acmrisk = mutator(alive_idx, working_acmrisk)
        # Add acceptable deviation in LNN for CPAP PAF not being age-specific
        acceptable_deviation = 0.01 if age_group_name == 'late_neonatal' else None
        fuzzy_check(working_acmrisk, acmrisk_targets, "ACMRisk after mutator", acceptable_deviation=acceptable_deviation)

    # Check that we've exactly replicated everything
    assert np.allclose(working_acmrisk, pipeline(alive_idx), rtol=0, atol=1e-14)

In [ ]:
step_by_step_check_acmrisk(0, LATE_NEONATAL_AGE_START)

### Check cause-specific mortality risks

In [ ]:
def get_csmrisk_targets(artifact_cause_name, age_group_start, age_group_end):
    return (
        art.load(f'cause.{artifact_cause_name}.mortality_risk')[f'draw_{draw_num}'].reset_index()
        .pipe(lambda df: df[(df.child_age_start == age_group_start) & (df.child_age_end == age_group_end)])
        .drop(columns=['child_age_start', 'child_age_end', 'year_start', 'year_end'])
        .set_index('sex_of_child')
        [f'draw_{draw_num}']
    )

In [ ]:
from vivarium_gates_mncnh.constants.data_values import PRETERM_DEATHS_DUE_TO_RDS_PROBABILITY

NEONATAL_CAUSES = [
    'neonatal_encephalopathy_due_to_birth_asphyxia_and_trauma',
    'neonatal_sepsis_and_other_neonatal_infections',
    'neonatal_preterm_birth',
    'neonatal_preterm_birth_with_rds',
    'neonatal_preterm_birth_without_rds',
]

CAUSE_NAME_TO_CSMRISK_PIPELINES = {}
for cause_name in NEONATAL_CAUSES:
    if cause_name == 'neonatal_preterm_birth':
        CAUSE_NAME_TO_CSMRISK_PIPELINES[cause_name] = [
            "neonatal_preterm_birth_with_rds.csmr",
            "neonatal_preterm_birth_without_rds.csmr",
        ]
    else:
        CAUSE_NAME_TO_CSMRISK_PIPELINES[cause_name] = [f"{cause_name}.csmr"]

def check_csmrisks(age_group_start, age_group_end):
    for cause_name in NEONATAL_CAUSES:
        print(cause_name)

        pipeline_names = CAUSE_NAME_TO_CSMRISK_PIPELINES[cause_name]
        csmrisk_values = 0
        for pipeline_name in pipeline_names:
            csmrisk_values += (
                main_sim.get_population(pipeline_name).loc[alive_idx]
            )

        artifact_cause_name = cause_name.replace('_with_rds', '').replace('_without_rds', '')
        csmrisk_targets = get_csmrisk_targets(artifact_cause_name, age_group_start, age_group_end)

        if cause_name == 'neonatal_preterm_birth_with_rds':
            csmrisk_targets *= PRETERM_DEATHS_DUE_TO_RDS_PROBABILITY
        elif cause_name == 'neonatal_preterm_birth_without_rds':
            csmrisk_targets *= (1 - PRETERM_DEATHS_DUE_TO_RDS_PROBABILITY)

        # Add acceptable deviation in LNN for CPAP PAF not being age-specific
        acceptable_deviation = (
            0.02
            if age_group_start == LATE_NEONATAL_AGE_START and cause_name in ('neonatal_preterm_birth', 'neonatal_preterm_birth_with_rds')
            else None
        )
        fuzzy_check(csmrisk_values, csmrisk_targets, f"early neonatal {cause_name} mortality risk", acceptable_deviation=acceptable_deviation)

check_csmrisks(0, LATE_NEONATAL_AGE_START)

#### Check frequency of negative other-causes mortality, and impact

This isn't the best check because it is only for a single draw and location. In the one I've chosen, this rarely happens, apparently.

In [ ]:
csmrisk_pipeline_names = sorted({item for values in CAUSE_NAME_TO_CSMRISK_PIPELINES.values() for item in values})
csmrisk_pipeline_names

In [ ]:
total_csmrisk = 0

for pipeline_name in csmrisk_pipeline_names:
    total_csmrisk += (
        main_sim.get_population(pipeline_name).loc[alive_idx]
    )

total_csmrisk

In [ ]:
total_csmrisk.describe()

In [ ]:
(total_csmrisk > mortality_risk).mean()

In [ ]:
# Expect CSMRisks to be underestimated by this amount as a result
(total_csmrisk - mortality_risk).clip(0).sum() / total_csmrisk.sum()

#### Check individual steps in calculation of cause-specific mortality risks

In [ ]:
def step_by_step_check_csmrisk(cause_name, age_group_start, age_group_end):
    artifact_cause_name = cause_name.replace('_with_rds', '').replace('_without_rds', '')
    csmrisk_targets = get_csmrisk_targets(artifact_cause_name, age_group_start, age_group_end)

    # https://github.com/ihmeuw/vivarium_gates_mncnh/blob/86f8e36e8f781549cd6671cba9ad33912638dc4a/src/vivarium_gates_mncnh/components/mortality.py#L357-L361
    print('Initial cause-specific mortality risk')
    component_name = ('preterm_birth.' if 'preterm_birth' in cause_name else 'neonatal_cause.') + cause_name
    component = main_sim_components[component_name]
    initial_csmrisk = component.csmrisk_table(alive_idx)
    # Does not vary except by sex
    assert (initial_csmrisk.groupby(pop.sex_of_child).nunique() == 1).all()
    # Exactly matches artifact values
    assert np.allclose(initial_csmrisk.groupby(pop.sex_of_child).mean(), csmrisk_targets, rtol=0, atol=1e-14)
    display(initial_csmrisk.groupby(pop.sex_of_child).mean())

    # https://github.com/ihmeuw/vivarium_gates_mncnh/blob/7886d2bcb71fd2c3e497997a2bcf44c43569b8ab/src/vivarium_gates_mncnh/components/neonatal_causes.py#L116-L136
    global any_failures
    if 'preterm_birth' in cause_name:
        # Next step (conceptually) is limiting to preterm babies
        print('Preterm prevalence:')
        prevalence = component.prevalence_table(alive_idx)
        # Does not vary except by sex
        assert (prevalence.groupby(pop.sex_of_child).nunique() == 1).all()
        display(prevalence.groupby(pop.sex_of_child).mean())

        ga_greater_than_37 = main_sim.get_population("gestational_age.exposure").loc[alive_idx] >= 37.0
        for sex in ['Female', 'Male']:
            try:
                fuzzy_checker.fuzzy_assert_proportion(
                    (~ga_greater_than_37)[pop.loc[alive_idx].sex_of_child == sex].sum(),
                    (pop.loc[alive_idx].sex_of_child == sex).sum(),
                    prevalence[pop.sex_of_child == sex].mean()
                )
            except AssertionError as e:
                print(e)
                any_failures = True
    
        limited_csmrisk_preterm = initial_csmrisk / prevalence
        limited_csmrisk_preterm.loc[ga_greater_than_37] = 0
        display(limited_csmrisk_preterm.groupby(pop.sex_of_child).mean())

        fuzzy_check(limited_csmrisk_preterm, csmrisk_targets, "CSMRisk after limiting to preterm")

        working_csmrisk = limited_csmrisk_preterm

        if cause_name == 'neonatal_preterm_birth_with_rds':
            working_csmrisk = working_csmrisk * PRETERM_DEATHS_DUE_TO_RDS_PROBABILITY
            csmrisk_targets = csmrisk_targets * PRETERM_DEATHS_DUE_TO_RDS_PROBABILITY
        elif cause_name == 'neonatal_preterm_birth_without_rds':
            working_csmrisk = working_csmrisk * (1 - PRETERM_DEATHS_DUE_TO_RDS_PROBABILITY)
            csmrisk_targets = csmrisk_targets * (1 - PRETERM_DEATHS_DUE_TO_RDS_PROBABILITY)
        
        fuzzy_check(working_csmrisk, csmrisk_targets, "CSMRisk after splitting to subcause")
    else:
        working_csmrisk = initial_csmrisk

    if age_group_start == 0 and age_group_end == LATE_NEONATAL_AGE_START:
        age_group_name = 'early_neonatal'
    elif age_group_start == LATE_NEONATAL_AGE_START and age_group_end == LATE_NEONATAL_AGE_END:
        age_group_name = 'late_neonatal'
    else:
        raise ValueError("Unknown age group")

    # Next (conceptually) is applying LBWSG PAF/normalizing constant and RR
    # TODO: Why is this different! Note that there is a PAF table on the risk effect, but it is not actually used!
    if 'preterm_birth' in cause_name:
        lbwsg_paf = component.paf_table(alive_idx)
    else:
        lbwsg_paf = main_sim_components[f'risk_effect.low_birth_weight_and_short_gestation_on_cause.{cause_name}.cause_specific_mortality_risk'].paf_table(alive_idx)
    lbwsg_rr = main_sim.get_population(f'effect_of_low_birth_weight_and_short_gestation_on_{age_group_name}_{cause_name}_relative_risk').loc[alive_idx]

    csmrisk_after_lbwsg = working_csmrisk * (1 - lbwsg_paf) * lbwsg_rr
    print('After LBWSG:')
    display(csmrisk_after_lbwsg.groupby(pop.sex_of_child).describe())

    fuzzy_check(csmrisk_after_lbwsg, csmrisk_targets, "CSMRisk after LBWSG")

    pipeline_names = CAUSE_NAME_TO_CSMRISK_PIPELINES[cause_name]
    assert len(pipeline_names) == 1
    pop_mgr = main_sim._builder.population._manager
    pipeline = pop_mgr._get_attribute_pipelines()[pipeline_names[0]]

    # Check that we've exactly replicated everything up to mutators
    assert np.allclose(csmrisk_after_lbwsg, pipeline.source(pop_mgr, alive_idx), rtol=0, atol=1e-14)

    # Now we add in intervention-based modifications
    print('Mutators:')
    display([m.name for m in pipeline.mutators])

    working_csmrisk = csmrisk_after_lbwsg

    for mutator in pipeline.mutators:
        print(f'After {mutator.name}')
        working_csmrisk = mutator(alive_idx, working_csmrisk)
        # Add acceptable deviation in LNN for CPAP PAF not being age-specific
        acceptable_deviation = (
            0.02
            if age_group_name == 'late_neonatal' and cause_name in ('neonatal_preterm_birth', 'neonatal_preterm_birth_with_rds')
            else None
        )
        fuzzy_check(working_csmrisk, csmrisk_targets, "CSMRisk after mutator", acceptable_deviation=acceptable_deviation)

    # Check that we've exactly replicated everything
    assert np.allclose(working_csmrisk, pipeline(alive_idx), rtol=0, atol=1e-14)

In [ ]:
for cause_name in NEONATAL_CAUSES:
    if cause_name == 'neonatal_preterm_birth':
        continue
    print(cause_name)
    step_by_step_check_csmrisk(cause_name, 0, LATE_NEONATAL_AGE_START)

## LNN mortality

### Step main sim to LNN mortality

In [ ]:
get_event_name = main_sim._builder.time.simulation_event_name()
get_event_name()

In [ ]:
%%time

while get_event_name() != 'late_neonatal_mortality':
    main_sim.step()
    print(get_event_name())

In [ ]:
assert get_event_name() == 'late_neonatal_mortality'

In [ ]:
# We now step to the *middle* of a time step
# https://github.com/ihmeuw/vivarium/blob/aeaa9e03ecbf67929a1ce94e934c2c14ececfa29/src/vivarium/framework/engine.py#L260-L268
self = main_sim
for event in self.time_step_events:
    self._lifecycle.set_state(event)
    pop_to_update = self._clock.get_active_simulants(
        self.get_population_index(),
        self._clock.event_time,
    )
    # https://github.com/ihmeuw/vivarium/blob/aeaa9e03ecbf67929a1ce94e934c2c14ececfa29/src/vivarium/framework/event.py#L119-L141
    clock = self._events.clock()
    step_size = self._events.step_size()
    event_time = clock + step_size

    e = Event(
        "time_step",
        pop_to_update,
        {},
        event_time,
        step_size,
    )

    listener = None
    listeners = self.time_step_emitters[event].__self__.listeners
    for priority_bucket in listeners:
        for listener in priority_bucket:
            if listener.__self__.__class__.__name__ == "NeonatalMortality" and event == "time_step":
                break
            listener(e)
        if listener is not None and listener.__self__.__class__.__name__ == "NeonatalMortality" and event == "time_step":
            break

    if listener is not None and listener.__self__.__class__.__name__ == "NeonatalMortality" and event == "time_step":
            break

In [ ]:
assert listener.__self__.__class__.__name__ == "NeonatalMortality" and event == "time_step"

In [ ]:
pop = main_sim.get_population([
    "child_alive",
    "child_age",
])
assert (pop[pop.child_alive == True].child_age >= LATE_NEONATAL_AGE_START).all()
assert (pop[pop.child_alive == True].child_age < LATE_NEONATAL_AGE_END).all()

In [ ]:
alive_idx = pop.index[pop['child_alive'] == True]

### Feed ENN PAFs back into the PAF sim

In [ ]:
acmr_paf_components = [k for k in paf_sim_components if 'risk_effect.low_birth_weight_' in k]
acmr_paf_components

In [ ]:
for component in acmr_paf_components:
    update_lookup_table(paf_sim_components[component].paf_table, acmrisk_pafs)

In [ ]:
# Check that our transfer actually worked
assert (
    set(paf_sim.get_population("all_causes.all_cause_mortality_risk.paf").loc[alive_idx])
    <=
    set(acmrisk_pafs['value'].unique())
)

### Step PAF sim to LNN

In [ ]:
%%time

assert (paf_sim.get_population("child_age") < LATE_NEONATAL_AGE_START).all()
paf_sim.step()

In [ ]:
paf_sim_pop = paf_sim.get_population([
    "child_alive",
    "child_age",
])
assert (paf_sim_pop[paf_sim_pop.child_alive == True].child_age >= LATE_NEONATAL_AGE_START).all()
assert (paf_sim_pop[paf_sim_pop.child_alive == True].child_age < LATE_NEONATAL_AGE_END).all()

### Transfer PAFs and preterm prevalence to main sim

In [ ]:
paf_observer = paf_sim_components['lbwsgpaf_observer.cause.all_causes.all_cause_mortality_risk']

In [ ]:
paf_sim._results.gather_results(Event(
    "time_step",
    index=paf_sim.get_population_index(),
    user_data={},
    time=paf_sim._clock.time,
    step_size=paf_sim._clock.step_size,
))

In [ ]:
acmrisk_pafs = process_pafs(
    paf_sim.get_results()['calculated_lbwsg_paf_on_cause.all_causes.all_cause_mortality_risk']
)
acmrisk_pafs

In [ ]:
preterm_csmr_pafs = process_pafs(
    paf_sim.get_results()['calculated_lbwsg_paf_on_cause.all_causes.all_cause_mortality_risk_preterm']
)
preterm_csmr_pafs

In [ ]:
acmr_paf_components = [k for k in main_sim_components if 'risk_effect.low_birth_weight_' in k and 'preterm' not in k]
acmr_paf_components

In [ ]:
for component in acmr_paf_components:
    update_lookup_table(main_sim_components[component].paf_table, acmrisk_pafs)

In [ ]:
preterm_paf_components = [k for k in main_sim_components if ('risk_effect.low_birth_weight_' in k and 'preterm' in k) or ('preterm_birth.' in k)]
preterm_paf_components

In [ ]:
for component in preterm_paf_components:
    update_lookup_table(main_sim_components[component].paf_table, preterm_csmr_pafs)

In [ ]:
late_neonatal_preterm_prevalence = paf_sim.get_results()['calculated_late_neonatal_preterm_prevalence'].assign(
    child_age_start=LATE_NEONATAL_AGE_START,
    child_age_end=LATE_NEONATAL_AGE_END,
    year_start=2023,
    year_end=2024,
).rename(columns={"child_sex": "sex_of_child"})
late_neonatal_preterm_prevalence

In [ ]:
preterm_prevalence_components = [k for k in main_sim_components if 'preterm_birth.' in k]

for component in preterm_prevalence_components:
    lookup_table = main_sim_components[component].prevalence_table
    enn_data = lookup_table.data[lookup_table.data.child_age_start == 0]
    update_lookup_table(lookup_table, pd.concat([enn_data, late_neonatal_preterm_prevalence], ignore_index=True))

In [ ]:
assert (
    set(main_sim_components['preterm_birth.neonatal_preterm_birth_with_rds'].prevalence_table(alive_idx).unique())
    <=
    set(late_neonatal_preterm_prevalence['value'].unique())
)

### Check all-cause mortality risk

In [ ]:
# Check that our transfer above actually worked
assert (
    set(main_sim.get_population("all_causes.all_cause_mortality_risk.paf").unique())
    <=
    set(acmrisk_pafs['value'].unique())
)

In [ ]:
# https://github.com/ihmeuw/vivarium_gates_mncnh/blob/9502fba51790f522523b8cd2925e539a692babf6/src/vivarium_gates_mncnh/components/mortality.py#L281-L283
pop = main_sim.get_population([
    "child_alive",
    "sex_of_child",
    "pregnancy_outcome",
    "effect_of_low_birth_weight_and_short_gestation_on_early_neonatal_all_causes_relative_risk",
    "effect_of_low_birth_weight_and_short_gestation_on_late_neonatal_all_causes_relative_risk"
])
alive_idx = pop.index[pop['child_alive'] == True]
mortality_risk = main_sim.get_population("death_in_age_group_probability").loc[alive_idx]

In [ ]:
acmrisk_targets = get_acmrisk_targets(LATE_NEONATAL_AGE_START, LATE_NEONATAL_AGE_END)
acmrisk_targets

In [ ]:
fuzzy_check(mortality_risk, acmrisk_targets, "late neonatal all-cause mortality risk", acceptable_deviation=0.01)

#### Check individual steps in calculation of all-cause mortality risk

In [ ]:
step_by_step_check_acmrisk(LATE_NEONATAL_AGE_START, LATE_NEONATAL_AGE_END)

### Check cause-specific mortality risks

In [ ]:
# Check that our transfer above actually worked
assert (
    set(main_sim.get_population("all_causes.all_cause_mortality_risk.paf").loc[alive_idx].unique())
    <=
    set(acmrisk_pafs['value'].unique())
)

In [ ]:
# Check that our transfer above actually worked
assert (
    set(main_sim_components['preterm_birth.neonatal_preterm_birth_with_rds'].lbwsg_acmr_paf(main_sim.get_population_index()).unique())
    <=
    set(preterm_csmr_pafs['value'].unique())
)

In [ ]:
check_csmrisks(LATE_NEONATAL_AGE_START, LATE_NEONATAL_AGE_END)

#### Check frequency of negative other-causes mortality, and impact

This isn't the best check because it is only for a single draw and location. In the one I've chosen, this rarely happens, apparently.

In [ ]:
csmrisk_pipeline_names = result = sorted({item for values in CAUSE_NAME_TO_CSMRISK_PIPELINES.values() for item in values})
csmrisk_pipeline_names

In [ ]:
total_csmrisk = 0

for pipeline_name in csmrisk_pipeline_names:
    total_csmrisk += (
        main_sim.get_population(pipeline_name).loc[alive_idx].rename('total_csmrisk')
    )

total_csmrisk

In [ ]:
total_csmrisk.describe()

In [ ]:
(total_csmrisk > mortality_risk).mean()

In [ ]:
# Expect CSMRisks to be underestimated by this amount as a result
(total_csmrisk - mortality_risk).clip(0).sum() / total_csmrisk.sum()

#### Check individual steps in calculation of cause-specific mortality risks

In [ ]:
for cause_name in NEONATAL_CAUSES:
    if cause_name == 'neonatal_preterm_birth':
        continue
    print(cause_name)
    step_by_step_check_csmrisk(cause_name, LATE_NEONATAL_AGE_START, LATE_NEONATAL_AGE_END)

In [ ]:
assert not any_failures

In [ ]:
!date